# 📦 Analyse du Chiffre d'Affaires — Projet E-Commerce
> 

Ce notebook réalise l'analyse complète des ventes : calcul du CA Brut, CA Net, TVA, identification du meilleur produit et export des résultats.

## ⚙️ Étape 1 — Importation des bibliothèques & chargement des données

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    'ID_Produit': [f'PROD{str(i).zfill(3)}' for i in range(1, 21)],
    'Nom_Produit': [
        'Laptop Pro', 'Smartphone X', 'Casque Audio', 'Tablette 10"', 'Clavier Mécanique',
        'Souris Ergonomique', 'Écran 27"', 'Webcam HD', 'SSD 1TB', 'RAM 16GB',
        'Imprimante Laser', 'Scanner A4', 'Routeur WiFi', 'Switch 8 ports', 'Câble HDMI',
        'Batterie externe', 'Hub USB-C', 'Tapis de souris XL', 'Lampe LED Bureau', 'Support Laptop'
    ],
    'Categorie': [
        'Informatique', 'Mobile', 'Audio', 'Mobile', 'Informatique',
        'Informatique', 'Informatique', 'Périphériques', 'Stockage', 'Composants',
        'Impression', 'Impression', 'Réseau', 'Réseau', 'Accessoires',
        'Accessoires', 'Accessoires', 'Accessoires', 'Accessoires', 'Accessoires'
    ],
    'Prix_Unitaire': [
        1299.99, 899.99, 149.99, 449.99, 129.99,
        79.99,   549.99,  89.99, 119.99,  89.99,
        349.99,  199.99,  99.99,  79.99,  19.99,
        49.99,    39.99,  24.99,  34.99,  29.99
    ],
    'Quantite': [
        15, 42, 78, 23, 55,
        92, 18, 67, 130, 200,
        12,  8, 45,  30, 250,
        180, 120, 95,  60,  85
    ],
    'Remise_Pct': [
        5, 10, 15,  8, 12,
        5,  7, 10, 20, 15,
        5,  0,  8,  5, 25,
        10, 15,  5,  0, 10
    ]
}

df = pd.DataFrame(data)
print(f'✅ Données chargées : {len(df)} produits, {len(df.columns)} colonnes')
df.head()

## 📦 Étape 2 — Calcul du Chiffre d'Affaires Brut
`CA_Brut = Prix_Unitaire × Quantité`

In [ ]:
df['CA_Brut'] = (df['Prix_Unitaire'] * df['Quantite']).round(2)
ca_brut_total = df['CA_Brut'].sum()

print(f'CA Brut Total : {ca_brut_total:,.2f} €')
df[['ID_Produit', 'Nom_Produit', 'Prix_Unitaire', 'Quantite', 'CA_Brut']]

### 📝 Interprétation — CA Brut
Le **CA Brut** représente le chiffre d'affaires théorique si aucune remise n'était accordée aux clients. C'est le **plafond maximal** de revenus que l'entreprise pourrait atteindre. Un CA Brut élevé reflète un bon volume de ventes et des prix unitaires compétitifs sur l'ensemble du catalogue.

## 💸 Étape 3 — Calcul du CA Net (après remises)
`CA_Net = CA_Brut × (1 - Remise%)`

In [ ]:
df['CA_Net'] = (df['CA_Brut'] * (1 - df['Remise_Pct'] / 100)).round(2)
ca_net_total  = df['CA_Net'].sum()
remise_totale = ca_brut_total - ca_net_total
taux_remise   = (remise_totale / ca_brut_total) * 100

print(f'Remises accordées : {remise_totale:,.2f} € ({taux_remise:.1f}% du CA Brut)')
print(f'CA Net Total      : {ca_net_total:,.2f} €')
df[['ID_Produit', 'Nom_Produit', 'CA_Brut', 'Remise_Pct', 'CA_Net']]

### 📝 Interprétation — CA Net
Le **CA Net** est le revenu réellement encaissé après déduction des remises commerciales. Les remises représentent environ **10,8% du CA Brut**, ce qui est **raisonnable et maîtrisé** (un taux inférieur à 15% indique une politique tarifaire saine). Le CA Net est l'indicateur de performance le plus fiable pour évaluer la santé financière réelle de l'entreprise.

## 🏦 Étape 4 — Calcul de la TVA (20%)
`TVA = CA_Net × 0.20`

In [ ]:
df['TVA_20pct'] = (df['CA_Net'] * 0.20).round(2)
tva_totale = df['TVA_20pct'].sum()

print(f'TVA Totale collectée : {tva_totale:,.2f} €')
df[['ID_Produit', 'Nom_Produit', 'CA_Net', 'TVA_20pct']]

### 📝 Interprétation — TVA
La TVA collectée (**32 457,47 €**) est perçue par l'entreprise **pour le compte de l'État**. Ce montant **ne constitue pas un revenu** pour l'entreprise : il devra être intégralement reversé à l'administration fiscale lors de la déclaration de TVA. Le CA Net hors TVA reste l'indicateur de revenu réel.

## ✅ Étape 5 — CA Total de l'entreprise

In [ ]:
ca_ttc_total = ca_net_total + tva_totale

resume = pd.DataFrame({
    'Indicateur': ['CA Brut Total', 'Remises accordées', 'CA Net Total', 'TVA collectée (20%)', 'CA TTC Total'],
    'Montant (€)': [ca_brut_total, -remise_totale, ca_net_total, tva_totale, ca_ttc_total]
})
resume['Montant (€)'] = resume['Montant (€)'].map(lambda x: f'{x:,.2f} €')
print('=== RÉCAPITULATIF FINANCIER ===')
resume

### 📝 Interprétation — CA Total
Le **CA Net total de 162 287,33 €** constitue la base de rentabilité de l'entreprise. Ces chiffres indiquent une activité commerciale **dynamique** avec un portefeuille de 20 produits répartis sur 9 catégories. La diversification assure une bonne résilience face aux fluctuations de la demande.

## 🏆 Étape 6 — Produit avec le plus gros CA Net

In [ ]:
idx_max = df['CA_Net'].idxmax()
best    = df.loc[idx_max]
pct_contribution = (best['CA_Net'] / ca_net_total) * 100

cat_summary = df.groupby('Categorie')['CA_Net'].sum().sort_values(ascending=False).reset_index()
cat_summary.columns = ['Catégorie', 'CA Net (€)']
cat_summary['CA Net (€)'] = cat_summary['CA Net (€)'].round(2)

print(f'🏆 Produit star : {best["Nom_Produit"]} ({best["ID_Produit"]})')
print(f'   CA Net       : {best["CA_Net"]:,.2f} € ({pct_contribution:.1f}% du CA total)')
print()
print('📦 CA Net par catégorie :')
cat_summary

### 📝 Interprétation — Meilleur produit
Le **Smartphone X (PROD002)** est le produit star de l'entreprise. Il génère à lui seul **21% du CA Net total**, ce qui en fait un pilier stratégique de l'activité. La catégorie **Mobile** est la plus performante (43 541 €). L'entreprise devrait prioriser le réapprovisionnement et la promotion de ces références clés pour maximiser les revenus.

## 📁 Étape 7 — Export du fichier `resultats_final.csv`

In [ ]:
df.to_csv('data/resultats_final.csv', index=False, encoding='utf-8-sig')

print('✅ Fichier exporté : data/resultats_final.csv')
print(f'   Lignes   : {len(df)} produits')
print(f'   Colonnes : {list(df.columns)}')
df

---
## 🏁 Conclusion générale

| Indicateur | Valeur |
|---|---|
| CA Brut Total | 181 898,95 € |
| Remises accordées | 19 611,62 € (10,8%) |
| **CA Net Total** | **162 287,33 €** |
| TVA collectée | 32 457,47 € |
| Produit star | Smartphone X — 34 019,62 € (21%) |
| Catégorie leader | Mobile — 43 541,41 € |

